# STIR-Net V1 — 28B Corrected Information-Sufficiency Causal Experiments

This notebook corrects the source-9 bookkeeping error in Notebook 28 and repeats the causal experiments on **all nine GT cells**.

## Critical correction

Notebook 28 incorrectly isolated source-9 proposal queries using:

```text
source_instance_id == 9
```

That excludes legitimate learned proposals whose anchors lie just outside the current binary component. In the current STIR-Net implementation those proposals intentionally have:

```text
source_instance_id = -1
```

and are still valid spatial proposals.

Notebook 28B therefore uses a **GT-aware geometric experimental pool only for this causal diagnostic**:

```text
all valid spatial-proposal queries
        ↓
keep source_id == 9
OR initial anchor near one of the 9 source-9 GT centers
        ↓
Hungarian match the pool to all 9 source-9 GT centers
        ↓
fixed 9 query ↔ 9 GT pairs
```

The GT-aware pool is **not proposed inference logic**. It only makes the causal experiment evaluate the intended nine biological cells instead of accidentally evaluating four.

## Additional corrections

- Experiment B generates overcomplete learned peaks directly from the proposal score field and filters them geometrically, so off-mask peaks are retained.
- Every downstream experiment asserts that all **9 unique GT cells** are represented.
- Coarse-mask probes train to a short convergence criterion rather than exactly 60 steps.
- Mask information experiments include controls:
  - query + relative XYZ only,
  - D0,
  - D0 + raw,
  - all explicit evidence,
  - all evidence + identical/common query,
  - all evidence with spatial evidence shuffled between cells,
  - oracle-center all evidence.
- A leave-one-cell-out head test checks whether high local-mask Dice comes from spatial evidence rather than merely memorizing the nine training masks.

This remains a **one-scene causal overfit diagnostic**, not a generalization benchmark.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import copy
import gc
import json
import math
import shutil
import time
import traceback

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.optimize import linear_sum_assignment

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.matcher import (
    build_local_support_masks,
    target_ids,
    target_masks_at_shape,
)
from learned.stirnet.model.query_builder import QUERY_SPATIAL_PROPOSAL
from learned.stirnet.model.types import StirNetOutput, TemporalState
from learned.stirnet.training.checkpoint import load_checkpoint

try:
    from learned.stirnet.training.trainer import move_batch_to_device
except ImportError:
    from learned.stirnet.training.trainer import move_to_device as move_batch_to_device


SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

# GT-aware diagnostic query-pool radii. We use the smallest radius that gives
# at least 9 proposal queries; source_id == 9 proposals are always retained.
QUERY_POOL_RADII_DREF = [0.75, 1.0, 1.25, 1.5, 2.0]

# Proposal-set experiment.
OVERCOMPLETE_NMS_OPTIONS_DREF = [0.20, 0.15, 0.10]
OVERCOMPLETE_POOL_RADII_DREF = [1.0, 1.25, 1.5, 2.0]
OVERCOMPLETE_MAX_GLOBAL_PEAKS = 512
PROPOSAL_REFINER_STEPS = 300
PROPOSAL_REFINER_LR = 2e-3
PROPOSAL_MAX_MOVE_DREF = 0.60

# Local evidence experiments.
LOCAL_GRID_SIZE = 20
LOCAL_EXTENT_DREF = 1.35
LOCAL_CENTER_STEPS = 250
LOCAL_MASK_STEPS = 120
LOCAL_HEAD_LR = 2e-3

# Dot-mask resolution/target experiment.
DOT_MASK_MAX_STEPS = 180
DOT_MASK_MIN_STEPS = 50
DOT_MASK_EVAL_EVERY = 10
DOT_MASK_PATIENCE_EVALS = 4
DOT_MASK_LR = 2e-3

# Leave-one-cell-out controls.
RUN_LOO_MASK_VALIDATION = True
LOO_MASK_STEPS = 80
LOO_VARIANTS = [
    "query_xyz",
    "explicit_all",
    "explicit_all_common_query",
    "explicit_all_shuffled",
]

SUPPORT_RADII_DREF = [1.0, 1.25, 1.5, 2.0, 2.5]

RUN_PROPOSAL_SET_EXPERIMENT = True
RUN_LOCAL_CENTER_EXPERIMENT = True
RUN_COARSE_MASK_PROBE = True
RUN_LOCAL_MASK_EXPERIMENT = True
RUN_SUPPORT_AUDIT = True
OPEN_NAPARI_AT_END = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
OVERNIGHT_ROOT = REPO_ROOT / "runs" / "stirnet" / "overnight"

BEST_SPATIAL_QUERY_CHECKPOINT = None

if BEST_SPATIAL_QUERY_CHECKPOINT is None:
    runs = sorted(
        OVERNIGHT_ROOT.glob("27_overnight_*"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    candidates = [
        run / "checkpoint_best_spatial_query.pt"
        for run in runs
        if (run / "checkpoint_best_spatial_query.pt").exists()
    ]
    if not candidates:
        raise FileNotFoundError(
            "No checkpoint_best_spatial_query.pt found. "
            "Set BEST_SPATIAL_QUERY_CHECKPOINT manually."
        )
    BEST_SPATIAL_QUERY_CHECKPOINT = candidates[0]
else:
    BEST_SPATIAL_QUERY_CHECKPOINT = Path(BEST_SPATIAL_QUERY_CHECKPOINT)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "experiments"
    / f"28B_corrected_information_sufficiency_{time.strftime('%Y%m%d_%H%M%S')}"
)
RUN_DIR.mkdir(parents=True, exist_ok=False)

LOG_PATH = RUN_DIR / "experiment.log"
RESULTS_JSONL = RUN_DIR / "results.jsonl"
ERRORS_JSONL = RUN_DIR / "errors.jsonl"

print("Repository :", REPO_ROOT)
print("Checkpoint :", BEST_SPATIAL_QUERY_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("Device     :", device)


In [ ]:
def now_text():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def _jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value):
        if value.numel() == 1:
            return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def log(message):
    line = f"[{now_text()}] {message}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")
        handle.flush()


def append_jsonl(path, payload):
    with path.open("a", encoding="utf-8") as handle:
        json.dump({k: _jsonable(v) for k, v in payload.items()}, handle)
        handle.write("\n")
        handle.flush()


def record_result(experiment, **kwargs):
    append_jsonl(
        RESULTS_JSONL,
        {
            "time": now_text(),
            "experiment": experiment,
            **kwargs,
        },
    )


def record_error(stage, exc):
    text = "".join(
        traceback.format_exception(
            type(exc),
            exc,
            exc.__traceback__,
        )
    )
    log(f"ERROR in {stage}: {type(exc).__name__}: {exc}")
    append_jsonl(
        ERRORS_JSONL,
        {
            "time": now_text(),
            "stage": stage,
            "type": type(exc).__name__,
            "message": str(exc),
            "traceback": text,
        },
    )


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


free_gib = shutil.disk_usage(REPO_ROOT.anchor).free / 1024**3
log(f"Free disk: {free_gib:.2f} GiB")

if free_gib < 2.0:
    raise RuntimeError("Less than 2 GiB free; refusing experiment run.")


## 1. Load the exact trained spatial-query checkpoint and source-9 scene

In [ ]:
if device.type != "cuda":
    raise RuntimeError("Notebook 28B requires CUDA.")

batch_cpu, sample_info = build_real_batch(DATA_DIR)
b = move_batch_to_device(batch_cpu, device)

b["spatial_inputs"] = b["spatial_inputs"].to(dtype=AMP_DTYPE)
b["instance_labels"] = b["instance_labels"].to(dtype=torch.int32)

targets = batch_cpu["targets"]
target = targets[0]

cfg = _reduced_config()
cfg.proposals.enabled = True
cfg.proposals.query_mode = "spatial_proposals"

model = StirNet(cfg).to(device)
checkpoint_info = load_checkpoint(
    BEST_SPATIAL_QUERY_CHECKPOINT,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)
model.eval()

gt_ids_all = target_ids(target).detach().cpu().long()
gt_centers_all = torch.as_tensor(
    target["centers_cellscale"],
    dtype=torch.float32,
)

current_labels_native = (
    batch_cpu["instance_labels"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

gt_labels_native = (
    torch.as_tensor(target["label_map"])
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

spacing_native = (
    batch_cpu["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)

dref_um = float(batch_cpu["dref_um"][0])

source9_gt_ids = np.unique(
    gt_labels_native[
        current_labels_native == SOURCE_ID
    ]
)
source9_gt_ids = source9_gt_ids[
    source9_gt_ids > 0
].astype(int)

source9_id_set = set(source9_gt_ids.tolist())

source9_gt_indices = torch.tensor(
    [
        row
        for row, gt_id in enumerate(gt_ids_all.tolist())
        if int(gt_id) in source9_id_set
    ],
    dtype=torch.long,
)

source9_gt_centers = (
    gt_centers_all[source9_gt_indices]
    .float()
)

if len(source9_gt_ids) != 9:
    raise RuntimeError(
        f"Expected exactly 9 source-9 GT cells; got {len(source9_gt_ids)}."
    )

if set(
    gt_ids_all[source9_gt_indices].tolist()
) != source9_id_set:
    raise AssertionError(
        "Source-9 GT-index mapping is inconsistent."
    )

log(
    f"Loaded checkpoint step={checkpoint_info.get('step')} | "
    f"source9 GT={source9_gt_ids.tolist()}"
)


## 2. Current spatial-only forward with intermediate tensors

In [ ]:
def make_empty_temporal(model, dtype):
    d_model = int(model.cfg.temporal.d_model)

    return TemporalState(
        tokens=torch.empty(
            (0, d_model),
            device=device,
            dtype=dtype,
        ),
        ref_um=torch.empty(
            (0, 3),
            device=device,
            dtype=torch.float32,
        ),
        ref_cellscale=torch.empty(
            (0, 3),
            device=device,
            dtype=torch.float32,
        ),
        salience=torch.empty(
            (0, 1),
            device=device,
            dtype=dtype,
        ),
        reliability=torch.empty(
            (0, 1),
            device=device,
            dtype=dtype,
        ),
        status=torch.empty(
            (0,),
            device=device,
            dtype=torch.long,
        ),
        edge_index=torch.empty(
            (2, 0),
            device=device,
            dtype=torch.long,
        ),
        edge_attr=torch.empty(
            (0, 22),
            device=device,
            dtype=torch.float32,
        ),
        batch_index=torch.empty(
            (0,),
            device=device,
            dtype=torch.long,
        ),
    )


def current_spatial_forward(model, token_cap=2048):
    old_cap = int(
        model.query_decoder.cfg.max_spatial_tokens
    )

    model.query_decoder.cfg.max_spatial_tokens = int(
        token_cap
    )
    model.cfg.decoder.max_spatial_tokens = int(
        token_cap
    )

    try:
        acq = model.acquisition(
            b["spacing_um"],
            b["dref_um"],
        )

        pyramid = model.encoder(
            b["spatial_inputs"],
            b["spacing_um"],
            acq,
            b.get("spatial_padding_mask"),
        )

        e3 = pyramid.features[3]

        e2 = model.decoder.decode_to_e2(
            e3,
            pyramid,
            acq,
        )

        d1, d0, native_mask_features = (
            model.decoder.decode_from_e2(
                e2,
                pyramid,
                acq,
            )
        )

        dense = model.dense_heads(d0)

        proposal_state, proposal_score_logits = (
            model.spatial_proposal_generator(
                d0,
                e2,
                b["spatial_inputs"],
                dense,
                b["instance_labels"],
                b["spacing_um"],
                pyramid.spacings_um[2],
                b["dref_um"],
                b["instance_ids"],
                b["instance_batch"],
                b["instance_centroids_um"],
                b.get("spatial_padding_mask"),
            )
        )

        dense = dict(dense)
        dense["proposal_score_logits"] = (
            proposal_score_logits
        )

        temporal = make_empty_temporal(
            model,
            e2.dtype,
        )

        q0 = model.query_builder(
            e2,
            pyramid.spacings_um[2],
            b["instance_labels"],
            b["instance_features"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b["dref_um"],
            temporal,
            memory_ablation="full",
            return_debug=False,
            full_attention=False,
            proposal_state=proposal_state,
            query_mode="spatial_proposals",
        )

        initial_refs = (
            q0.references_cellscale.clone()
        )

        qf, decoder_outputs = (
            model.query_decoder(
                q0,
                [e3, e2, d1],
                [
                    pyramid.spacings_um[3],
                    pyramid.spacings_um[2],
                    pyramid.spacings_um[1],
                ],
                b["instance_labels"],
                b["dref_um"],
                temporal,
                memory_ablation="full",
                return_debug=False,
                full_attention=False,
            )
        )

        final = decoder_outputs[-1]

        output = StirNetOutput(
            exist_logits=final["exist_logits"],
            centers_cellscale=final[
                "centers_cellscale"
            ],
            coarse_mask_logits=final[
                "coarse_mask_logits"
            ],
            coarse_spacing_um=final[
                "coarse_spacing_um"
            ],
            query_embeddings=qf.embeddings,
            native_mask_embeddings=(
                model.native_mask_head(
                    qf.embeddings
                )
            ),
            query_types=qf.query_types,
            query_padding_mask=qf.padding_mask,
            source_instance_ids=(
                qf.source_instance_ids
            ),
            query_initial_references_cellscale=(
                initial_refs
            ),
            temporal_salience=qf.temporal_salience,
            temporal_reliability=(
                qf.temporal_reliability
            ),
            aux_outputs=decoder_outputs[:-1],
            dense_outputs=dense,
            mask_features=native_mask_features,
            spacing_um=b["spacing_um"],
            dref_um=b["dref_um"],
            instance_labels=b["instance_labels"],
            debug=None,
            proposals=proposal_state,
        )

        intermediate = {
            "pyramid": pyramid,
            "e3": e3,
            "e2": e2,
            "d1": d1,
            "d0": d0,
            "dense": dense,
            "proposals": proposal_state,
            "q0": q0,
            "qf": qf,
            "decoder_outputs": decoder_outputs,
        }

        return output, intermediate

    finally:
        model.query_decoder.cfg.max_spatial_tokens = (
            old_cap
        )
        model.cfg.decoder.max_spatial_tokens = old_cap


torch.cuda.reset_peak_memory_stats()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    baseline_outputs, baseline_intermediate = (
        current_spatial_forward(
            model,
            token_cap=2048,
        )
    )

log(
    "Baseline forward OK | "
    f"coarse shape={tuple(baseline_outputs.coarse_mask_logits.shape[-3:])} | "
    f"peak={torch.cuda.max_memory_allocated()/1024**3:.2f} GiB"
)


## 3. Correct source-9 experimental query pool

This is the key correction.

A spatial proposal can represent one of the nine source-9 GT cells even when its anchor falls just outside the current source-9 mask and therefore has `source_instance_id = -1`.

For this **GT-aware diagnostic only**, we retain every valid spatial-proposal query that either:

- has `source_instance_id == 9`, or
- has an initial anchor within a small physical radius of any of the nine source-9 GT centers.

The radius is increased only until at least nine candidates are available.

Then a one-to-one Hungarian assignment on **initial anchor distance** chooses the fixed nine query↔GT pairs used by Experiments C–F.


In [ ]:
def center_set_metrics(refs, gt_centers, name):
    refs = refs.detach().float().cpu()
    gt = gt_centers.detach().float().cpu()

    if len(refs) == 0:
        raise ValueError(f"{name}: empty reference set")

    dist = torch.cdist(
        refs,
        gt,
        p=2,
    )

    gt_nearest = dist.min(dim=0).values
    query_nearest_gt = dist.argmin(dim=1)

    nearest_counts = torch.bincount(
        query_nearest_gt,
        minlength=len(gt),
    )

    row_np, col_np = linear_sum_assignment(
        dist.numpy()
    )

    matched = dist[
        torch.as_tensor(row_np),
        torch.as_tensor(col_np),
    ]

    return {
        "name": name,
        "query_count": int(len(refs)),
        "recall_0p5": float(
            (gt_nearest <= 0.5)
            .float()
            .mean()
        ),
        "recall_1p0": float(
            (gt_nearest <= 1.0)
            .float()
            .mean()
        ),
        "hungarian_mean_dref": float(
            matched.mean()
        ),
        "hungarian_max_dref": float(
            matched.max()
        ),
        "hungarian_mean_um": float(
            matched.mean() * dref_um
        ),
        "duplicate_nearest_gt_count": int(
            (nearest_counts > 1).sum()
        ),
        "missing_gt_0p5": int(
            (gt_nearest > 0.5).sum()
        ),
    }


def build_source9_query_pool(
    outputs,
):
    valid = (
        ~outputs.query_padding_mask[0]
        .detach()
        .cpu()
    )

    proposal_type = (
        outputs.query_types[0]
        .detach()
        .cpu()
        == QUERY_SPATIAL_PROPOSAL
    )

    all_slots = torch.nonzero(
        valid & proposal_type,
        as_tuple=False,
    ).flatten()

    if len(all_slots) == 0:
        raise RuntimeError(
            "No valid spatial-proposal queries."
        )

    refs = (
        outputs.query_initial_references_cellscale[
            0,
            all_slots.to(device),
        ]
        .detach()
        .float()
        .cpu()
    )

    source_ids = (
        outputs.source_instance_ids[
            0,
            all_slots.to(device),
        ]
        .detach()
        .cpu()
        .long()
    )

    distance_to_s9 = torch.cdist(
        refs,
        source9_gt_centers.float(),
    )

    min_distance = (
        distance_to_s9.min(dim=1).values
    )

    selected_local = None
    selected_radius = None

    for radius in QUERY_POOL_RADII_DREF:
        eligible = (
            (source_ids == SOURCE_ID)
            | (min_distance <= float(radius))
        )

        local_rows = torch.nonzero(
            eligible,
            as_tuple=False,
        ).flatten()

        if len(local_rows) >= 9:
            selected_local = local_rows
            selected_radius = radius
            break

    if selected_local is None:
        raise RuntimeError(
            "Could not build a source-9 proposal-query pool "
            f"with at least 9 candidates. "
            f"Available valid proposals={len(all_slots)}, "
            f"max nearby at {QUERY_POOL_RADII_DREF[-1]} dref="
            f"{int((min_distance <= QUERY_POOL_RADII_DREF[-1]).sum())}, "
            f"source_id==9={int((source_ids == SOURCE_ID).sum())}."
        )

    pool_slots = all_slots[selected_local]
    pool_refs = refs[selected_local]
    pool_sources = source_ids[selected_local]
    pool_min_distance = min_distance[
        selected_local
    ]

    return {
        "slots": pool_slots,
        "refs": pool_refs,
        "source_ids": pool_sources,
        "min_distance": pool_min_distance,
        "radius": float(selected_radius),
    }


source9_pool = build_source9_query_pool(
    baseline_outputs
)

pool_slots = source9_pool["slots"]
pool_initial_refs = source9_pool["refs"]

pool_final_refs = (
    baseline_outputs.centers_cellscale[
        0,
        pool_slots.to(device),
    ]
    .detach()
    .float()
    .cpu()
)

pool_dist = torch.cdist(
    pool_initial_refs,
    source9_gt_centers.float(),
)

match_pool_rows_np, match_gt_rows_np = (
    linear_sum_assignment(
        pool_dist.numpy()
    )
)

match_pool_rows = torch.as_tensor(
    match_pool_rows_np,
    dtype=torch.long,
)

selected_gt_rows = torch.as_tensor(
    match_gt_rows_np,
    dtype=torch.long,
)

selected_query_slots = pool_slots[
    match_pool_rows
]

selected_initial_refs = pool_initial_refs[
    match_pool_rows
]

selected_final_refs = pool_final_refs[
    match_pool_rows
]

selected_gt_centers = (
    source9_gt_centers[
        selected_gt_rows
    ]
)

selected_gt_ids = (
    source9_gt_ids[
        selected_gt_rows.numpy()
    ]
)

if len(selected_query_slots) != 9:
    raise AssertionError(
        f"Expected exactly 9 selected query slots, got {len(selected_query_slots)}."
    )

if len(np.unique(selected_gt_ids)) != 9:
    raise AssertionError(
        "The fixed query↔GT pairing does not contain 9 unique GT cells."
    )

if set(selected_gt_ids.tolist()) != set(
    source9_gt_ids.tolist()
):
    raise AssertionError(
        "The fixed pairing does not cover all source-9 GT cells."
    )

selection_audit = pd.DataFrame(
    {
        "query_slot": selected_query_slots.numpy(),
        "gt_id": selected_gt_ids,
        "source_instance_id": (
            baseline_outputs.source_instance_ids[
                0,
                selected_query_slots.to(device),
            ]
            .detach()
            .cpu()
            .numpy()
        ),
        "initial_distance_dref": (
            torch.linalg.vector_norm(
                selected_initial_refs
                - selected_gt_centers,
                dim=-1,
            )
            .numpy()
        ),
        "final_distance_dref": (
            torch.linalg.vector_norm(
                selected_final_refs
                - selected_gt_centers,
                dim=-1,
            )
            .numpy()
        ),
    }
)

display(selection_audit)

print(
    "Experimental pool size:",
    len(pool_slots),
)
print(
    "Pool geometric radius used:",
    source9_pool["radius"],
    "dref",
)
print(
    "Selected off-mask queries:",
    int(
        (
            selection_audit[
                "source_instance_id"
            ]
            < 0
        ).sum()
    ),
    "/ 9",
)

selection_audit.to_csv(
    RUN_DIR / "00_source9_selection_audit.csv",
    index=False,
)

record_result(
    "00_source9_selection",
    pool_size=len(pool_slots),
    pool_radius_dref=source9_pool["radius"],
    selected_query_count=len(
        selected_query_slots
    ),
    selected_gt_count=len(
        np.unique(selected_gt_ids)
    ),
    selected_off_mask_count=int(
        (
            selection_audit[
                "source_instance_id"
            ]
            < 0
        ).sum()
    ),
)


## Experiment A — Initial vs final center behavior on the corrected pool

In [ ]:
initial_pool_summary = center_set_metrics(
    pool_initial_refs,
    source9_gt_centers,
    "initial_pool",
)

final_pool_summary = center_set_metrics(
    pool_final_refs,
    source9_gt_centers,
    "final_pool",
)

paired_initial_error = (
    torch.linalg.vector_norm(
        selected_initial_refs
        - selected_gt_centers,
        dim=-1,
    )
)

paired_final_error = (
    torch.linalg.vector_norm(
        selected_final_refs
        - selected_gt_centers,
        dim=-1,
    )
)

paired_movement = (
    torch.linalg.vector_norm(
        selected_final_refs
        - selected_initial_refs,
        dim=-1,
    )
)

center_audit = pd.DataFrame(
    [
        initial_pool_summary,
        final_pool_summary,
    ]
)

display(center_audit)

paired_center_df = pd.DataFrame(
    {
        "query_slot": (
            selected_query_slots.numpy()
        ),
        "gt_id": selected_gt_ids,
        "initial_error_dref": (
            paired_initial_error.numpy()
        ),
        "final_error_dref": (
            paired_final_error.numpy()
        ),
        "query_movement_dref": (
            paired_movement.numpy()
        ),
    }
)

display(paired_center_df)

print(
    "Paired initial mean:",
    float(paired_initial_error.mean()),
    "dref",
)
print(
    "Paired final mean:",
    float(paired_final_error.mean()),
    "dref",
)

center_audit.to_csv(
    RUN_DIR / "A_center_pool_audit.csv",
    index=False,
)

paired_center_df.to_csv(
    RUN_DIR / "A_center_paired_9cells.csv",
    index=False,
)

record_result(
    "A_center_audit",
    initial_pool=initial_pool_summary,
    final_pool=final_pool_summary,
    paired_initial_mean_dref=float(
        paired_initial_error.mean()
    ),
    paired_final_mean_dref=float(
        paired_final_error.mean()
    ),
    movement_mean_dref=float(
        paired_movement.mean()
    ),
)


## Experiment B — Proposal decision logic with off-mask candidates retained

Instead of using `source_instance_id`, this phase works directly from the learned proposal-score field.

It:

1. extracts overcomplete learned maxima with smaller NMS;
2. keeps any peak geometrically near the nine source-9 GT centers;
3. computes the **real current proposal-local embedding** at those anchors;
4. compares:
   - independent per-candidate refinement,
   - proposal-set Transformer refinement.

Both models get the same candidate embedding, score, and position.


In [ ]:
class IndependentProposalRefiner(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden=64,
    ):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden,
            ),
            nn.SiLU(),
            nn.Linear(
                hidden,
                hidden,
            ),
            nn.SiLU(),
        )

        self.exist = nn.Linear(
            hidden,
            1,
        )

        self.delta = nn.Linear(
            hidden,
            3,
        )

    def forward(self, x, refs):
        h = self.body(x)

        exist = (
            self.exist(h)
            .squeeze(-1)
        )

        refined = refs + (
            torch.tanh(
                self.delta(h)
            )
            * PROPOSAL_MAX_MOVE_DREF
        )

        return exist, refined


class ProposalSetRefiner(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=64,
    ):
        super().__init__()

        self.input_proj = nn.Linear(
            input_dim,
            d_model,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=128,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = (
            nn.TransformerEncoder(
                layer,
                num_layers=2,
            )
        )

        self.exist = nn.Linear(
            d_model,
            1,
        )

        self.delta = nn.Linear(
            d_model,
            3,
        )

    def forward(self, x, refs):
        h = self.encoder(
            self.input_proj(x)[None]
        )[0]

        exist = (
            self.exist(h)
            .squeeze(-1)
        )

        refined = refs + (
            torch.tanh(
                self.delta(h)
            )
            * PROPOSAL_MAX_MOVE_DREF
        )

        return exist, refined


def proposal_refiner_loss(
    exist_logits,
    refined_refs,
    gt_centers,
):
    distance = torch.cdist(
        refined_refs.float(),
        gt_centers.float(),
    )

    rows_np, cols_np = (
        linear_sum_assignment(
            distance.detach()
            .cpu()
            .numpy()
        )
    )

    rows = torch.as_tensor(
        rows_np,
        device=device,
        dtype=torch.long,
    )

    cols = torch.as_tensor(
        cols_np,
        device=device,
        dtype=torch.long,
    )

    target_exist = torch.zeros_like(
        exist_logits
    )
    target_exist[rows] = 1.0

    negative_count = max(
        0,
        len(exist_logits)
        - len(rows),
    )

    pos_weight = torch.tensor(
        max(
            1.0,
            negative_count
            / max(len(rows), 1),
        ),
        device=device,
    )

    exist_loss = (
        F.binary_cross_entropy_with_logits(
            exist_logits,
            target_exist,
            pos_weight=pos_weight,
        )
    )

    center_loss = F.smooth_l1_loss(
        refined_refs[rows],
        gt_centers[cols],
        beta=0.10,
    )

    return (
        exist_loss
        + 4.0 * center_loss
    )


def evaluate_refiner(
    exist_logits,
    refined_refs,
    name,
):
    topk = torch.topk(
        exist_logits,
        k=9,
    ).indices

    selected = refined_refs[
        topk
    ]

    summary = center_set_metrics(
        selected,
        source9_gt_centers,
        name,
    )

    summary[
        "mean_selected_exist_prob"
    ] = float(
        exist_logits[
            topk
        ]
        .sigmoid()
        .mean()
        .detach()
        .cpu()
    )

    return summary


def build_overcomplete_source9_candidates():
    generator = (
        model.spatial_proposal_generator
    )

    score_logits = (
        baseline_intermediate[
            "dense"
        ][
            "proposal_score_logits"
        ]
    )

    spacing = b[
        "spacing_um"
    ][0]

    dref = b[
        "dref_um"
    ][0]

    padding = (
        None
        if b.get(
            "spatial_padding_mask"
        ) is None
        else b[
            "spatial_padding_mask"
        ][0]
    )

    chosen = None

    old_nms = float(
        generator.cfg.nms_radius_dref
    )

    try:
        for nms_radius in (
            OVERCOMPLETE_NMS_OPTIONS_DREF
        ):
            generator.cfg.nms_radius_dref = (
                float(nms_radius)
            )

            voxels = generator._learned_centers(
                score_logits[
                    0, 0
                ],
                spacing,
                dref,
                padding,
                OVERCOMPLETE_MAX_GLOBAL_PEAKS,
            )

            refs_um = (
                generator
                ._relative_voxel_centers_um(
                    voxels,
                    tuple(
                        int(v)
                        for v
                        in score_logits.shape[-3:]
                    ),
                    spacing,
                )
            )

            refs_cellscale = (
                refs_um
                / dref.clamp_min(
                    1e-8
                )
            )

            if len(
                refs_cellscale
            ) == 0:
                continue

            distances = torch.cdist(
                refs_cellscale.float(),
                source9_gt_centers
                .to(device)
                .float(),
            )

            min_distance = (
                distances
                .min(dim=1)
                .values
            )

            for pool_radius in (
                OVERCOMPLETE_POOL_RADII_DREF
            ):
                keep = (
                    min_distance
                    <= float(
                        pool_radius
                    )
                )

                rows = torch.nonzero(
                    keep,
                    as_tuple=False,
                ).flatten()

                if len(rows) >= 9:
                    chosen = {
                        "nms_radius": (
                            float(
                                nms_radius
                            )
                        ),
                        "pool_radius": (
                            float(
                                pool_radius
                            )
                        ),
                        "refs_um": (
                            refs_um[
                                rows
                            ]
                        ),
                        "refs_cellscale": (
                            refs_cellscale[
                                rows
                            ]
                        ),
                    }
                    break

            if chosen is not None:
                break

    finally:
        generator.cfg.nms_radius_dref = (
            old_nms
        )

    if chosen is None:
        raise RuntimeError(
            "Unable to generate >=9 learned source-9 "
            "candidates even after relaxed NMS and "
            "GT-geometric pooling."
        )

    refs_um = chosen[
        "refs_um"
    ]

    embeddings, scores = (
        generator._local_embeddings(
            0,
            refs_um,
            baseline_intermediate[
                "d0"
            ],
            baseline_intermediate[
                "e2"
            ],
            b["spatial_inputs"],
            baseline_intermediate[
                "dense"
            ],
            b["spacing_um"][0],
            baseline_intermediate[
                "pyramid"
            ].spacings_um[2][0],
            b["dref_um"][0],
            score_logits,
        )
    )

    chosen[
        "embeddings"
    ] = embeddings.detach().float()

    chosen[
        "scores"
    ] = scores.detach().float()

    return chosen


proposal_set_results = []

if RUN_PROPOSAL_SET_EXPERIMENT:
    try:
        overcomplete = (
            build_overcomplete_source9_candidates()
        )

        candidate_refs = (
            overcomplete[
                "refs_cellscale"
            ]
            .detach()
            .float()
        )

        candidate_features = torch.cat(
            [
                overcomplete[
                    "embeddings"
                ],
                overcomplete[
                    "scores"
                ][:, None],
                candidate_refs,
            ],
            dim=-1,
        )

        gt_centers_device = (
            source9_gt_centers
            .to(device)
            .float()
        )

        raw_summary = (
            center_set_metrics(
                candidate_refs,
                gt_centers_device,
                "overcomplete_raw_candidates",
            )
        )

        raw_summary[
            "model"
        ] = "raw_candidates"

        raw_summary[
            "nms_radius_dref"
        ] = overcomplete[
            "nms_radius"
        ]

        raw_summary[
            "pool_radius_dref"
        ] = overcomplete[
            "pool_radius"
        ]

        proposal_set_results.append(
            raw_summary
        )

        for model_name, refiner_cls in [
            (
                "independent_refiner",
                IndependentProposalRefiner,
            ),
            (
                "set_refiner",
                ProposalSetRefiner,
            ),
        ]:
            torch.manual_seed(SEED)

            refiner = refiner_cls(
                candidate_features.shape[-1]
            ).to(device)

            optimizer = torch.optim.AdamW(
                refiner.parameters(),
                lr=PROPOSAL_REFINER_LR,
                weight_decay=1e-4,
            )

            best_rank = None
            best_summary = None

            for step in range(
                PROPOSAL_REFINER_STEPS + 1
            ):
                refiner.train()

                optimizer.zero_grad(
                    set_to_none=True
                )

                exist, refined = refiner(
                    candidate_features,
                    candidate_refs,
                )

                loss = proposal_refiner_loss(
                    exist,
                    refined,
                    gt_centers_device,
                )

                if step < PROPOSAL_REFINER_STEPS:
                    loss.backward()

                    torch.nn.utils.clip_grad_norm_(
                        refiner.parameters(),
                        1.0,
                    )

                    optimizer.step()

                if (
                    step == 0
                    or step % 50 == 0
                    or step
                    == PROPOSAL_REFINER_STEPS
                ):
                    refiner.eval()

                    with torch.no_grad():
                        exist_eval, refs_eval = (
                            refiner(
                                candidate_features,
                                candidate_refs,
                            )
                        )

                        summary = (
                            evaluate_refiner(
                                exist_eval,
                                refs_eval,
                                (
                                    f"{model_name}"
                                    f"_step_{step}"
                                ),
                            )
                        )

                    summary[
                        "loss"
                    ] = float(
                        loss.detach().cpu()
                    )

                    log(
                        f"B {model_name} step={step} | "
                        f"recall0.5={summary['recall_0p5']:.3f} | "
                        f"mean={summary['hungarian_mean_dref']:.3f} | "
                        f"dup={summary['duplicate_nearest_gt_count']}"
                    )

                    rank = (
                        summary[
                            "recall_0p5"
                        ],
                        -summary[
                            "missing_gt_0p5"
                        ],
                        -summary[
                            "duplicate_nearest_gt_count"
                        ],
                        -summary[
                            "hungarian_mean_dref"
                        ],
                    )

                    if (
                        best_rank is None
                        or rank > best_rank
                    ):
                        best_rank = rank
                        best_summary = dict(
                            summary
                        )

            best_summary[
                "model"
            ] = model_name

            best_summary[
                "nms_radius_dref"
            ] = overcomplete[
                "nms_radius"
            ]

            best_summary[
                "pool_radius_dref"
            ] = overcomplete[
                "pool_radius"
            ]

            proposal_set_results.append(
                best_summary
            )

            del refiner, optimizer
            cleanup()

        proposal_set_df = pd.DataFrame(
            proposal_set_results
        )

        display(
            proposal_set_df
        )

        proposal_set_df.to_csv(
            RUN_DIR
            / "B_proposal_set_reasoning.csv",
            index=False,
        )

        record_result(
            "B_proposal_set_reasoning",
            rows=(
                proposal_set_df
                .to_dict("records")
            ),
        )

    except Exception as exc:
        record_error(
            "B_proposal_set_reasoning",
            exc,
        )

        proposal_set_df = pd.DataFrame(
            proposal_set_results
        )

        cleanup()

else:
    proposal_set_df = pd.DataFrame()


## 4. Shared high-resolution local-evidence sampler

Experiments C and E use a local physical cube around each fixed query anchor.

The tensor can contain:

```text
D0 learned feature
raw
current mask
EDT
boundary input
marker
dense foreground probability
dense center probability
dense boundary probability
relative z/y/x
```

No repository weights are changed.


In [ ]:
def physical_sampling_grid(
    ref_cellscale,
    feature_shape,
    spacing_um,
    dref,
    grid_size=LOCAL_GRID_SIZE,
    extent_dref=LOCAL_EXTENT_DREF,
):
    axis = torch.linspace(
        -extent_dref,
        extent_dref,
        grid_size,
        device=device,
        dtype=torch.float32,
    ) * dref.float()

    offsets = torch.stack(
        torch.meshgrid(
            axis,
            axis,
            axis,
            indexing="ij",
        ),
        dim=-1,
    )

    center_um = (
        ref_cellscale.float()
        * dref.float()
    )

    points_zyx_um = (
        center_um[
            None,
            None,
            None,
            :
        ]
        + offsets
    )

    extent_um = torch.tensor(
        [
            feature_shape[0] - 1,
            feature_shape[1] - 1,
            feature_shape[2] - 1,
        ],
        device=device,
        dtype=torch.float32,
    ) * spacing_um.float()

    normalized_zyx = (
        points_zyx_um
        / (
            0.5
            * extent_um
        ).clamp_min(1e-8)
    )

    return normalized_zyx[
        ...,
        [2, 1, 0],
    ]


@torch.no_grad()
def sample_feature_cubes(
    feature,
    refs_cellscale,
    spacing_um,
    dref,
):
    cubes = []

    for ref in refs_cellscale:
        grid = physical_sampling_grid(
            ref,
            tuple(
                int(v)
                for v
                in feature.shape[-3:]
            ),
            spacing_um,
            dref,
        )[None]

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            sampled = F.grid_sample(
                feature,
                grid,
                mode="bilinear",
                padding_mode="zeros",
                align_corners=True,
            )[0]

        cubes.append(
            sampled.float()
        )

    return torch.stack(
        cubes,
        dim=0,
    )


def relative_xyz_channels(count):
    axis = torch.linspace(
        -LOCAL_EXTENT_DREF,
        LOCAL_EXTENT_DREF,
        LOCAL_GRID_SIZE,
        device=device,
        dtype=torch.float32,
    )

    zz, yy, xx = torch.meshgrid(
        axis,
        axis,
        axis,
        indexing="ij",
    )

    rel = torch.stack(
        [zz, yy, xx],
        dim=0,
    )

    return rel[
        None
    ].expand(
        count,
        -1,
        -1,
        -1,
        -1,
    )


@torch.no_grad()
def build_explicit_evidence(
    refs_cellscale,
):
    d0_cubes = sample_feature_cubes(
        baseline_intermediate[
            "d0"
        ],
        refs_cellscale,
        b["spacing_um"][0],
        b["dref_um"][0],
    )

    input_cubes = sample_feature_cubes(
        b["spatial_inputs"],
        refs_cellscale,
        b["spacing_um"][0],
        b["dref_um"][0],
    )

    dense_stack = torch.cat(
        [
            baseline_intermediate[
                "dense"
            ][
                "foreground_logits"
            ].sigmoid(),
            baseline_intermediate[
                "dense"
            ][
                "center_heatmap_logits"
            ].sigmoid(),
            baseline_intermediate[
                "dense"
            ][
                "boundary_logits"
            ].sigmoid(),
        ],
        dim=1,
    )

    dense_cubes = sample_feature_cubes(
        dense_stack,
        refs_cellscale,
        b["spacing_um"][0],
        b["dref_um"][0],
    )

    return {
        "d0": d0_cubes,
        "inputs": input_cubes,
        "dense": dense_cubes,
        "rel": relative_xyz_channels(
            len(refs_cellscale)
        ),
    }


@torch.no_grad()
def sample_gt_cubes(
    gt_ids_ordered,
    refs_cellscale,
):
    cubes = []

    for gt_id, ref in zip(
        gt_ids_ordered,
        refs_cellscale,
    ):
        native = torch.from_numpy(
            (
                gt_labels_native
                == int(gt_id)
            ).astype(
                np.float32
            )
        )[
            None,
            None,
        ].to(device)

        grid = physical_sampling_grid(
            ref,
            tuple(
                int(v)
                for v
                in native.shape[-3:]
            ),
            b["spacing_um"][0],
            b["dref_um"][0],
        )[None]

        sampled = F.grid_sample(
            native,
            grid,
            mode="nearest",
            padding_mode="zeros",
            align_corners=True,
        )[0, 0]

        cubes.append(sampled)

        del native

    cleanup()

    return torch.stack(
        cubes,
        dim=0,
    )


## Experiment C — Does precise center localization need direct local 3-D evidence?

All four probes start from the **same nine initial query anchors** and are trained against the same nine GT centers:

1. `query_only`: new MLP from the existing query vector.
2. `local_tensor_only`: direct local 3-D evidence, no query vector.
3. `local_tensor_plus_query`: both.
4. `shuffled_tensor_plus_query`: each query receives another cell's local tensor.

Interpretation:

- If query-only fits just as well, the query already contains sufficient localization information and the current center-head/training is the issue.
- If local tensor only or tensor+query is materially better, precise geometry benefits from direct spatial evidence.
- If shuffled tensors remain equally good, the apparent local-tensor gain may just be memorization through query identity.


In [ ]:
class QueryOnlyCenterProbe(nn.Module):
    def __init__(
        self,
        query_dim,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                query_dim,
                64,
            ),
            nn.SiLU(),
            nn.Linear(
                64,
                64,
            ),
            nn.SiLU(),
            nn.Linear(
                64,
                3,
            ),
        )

    def forward(
        self,
        evidence,
        query,
    ):
        return (
            torch.tanh(
                self.net(query)
            )
            * 0.75
        )


class TensorOnlyCenterProbe(nn.Module):
    def __init__(
        self,
        in_channels,
        query_dim,
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv3d(
                in_channels,
                24,
                3,
                padding=1,
            ),
            nn.GroupNorm(
                4,
                24,
            ),
            nn.SiLU(),
            nn.Conv3d(
                24,
                32,
                3,
                padding=1,
            ),
            nn.GroupNorm(
                4,
                32,
            ),
            nn.SiLU(),
            nn.AdaptiveAvgPool3d(1),
        )

        self.out = nn.Sequential(
            nn.Linear(
                32,
                64,
            ),
            nn.SiLU(),
            nn.Linear(
                64,
                3,
            ),
        )

    def forward(
        self,
        evidence,
        query,
    ):
        h = (
            self.conv(
                evidence
            )
            .flatten(1)
        )

        return (
            torch.tanh(
                self.out(h)
            )
            * 0.75
        )


class TensorQueryCenterProbe(nn.Module):
    def __init__(
        self,
        in_channels,
        query_dim,
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv3d(
                in_channels,
                24,
                3,
                padding=1,
            ),
            nn.GroupNorm(
                4,
                24,
            ),
            nn.SiLU(),
            nn.Conv3d(
                24,
                32,
                3,
                padding=1,
            ),
            nn.GroupNorm(
                4,
                32,
            ),
            nn.SiLU(),
            nn.AdaptiveAvgPool3d(1),
        )

        self.query = nn.Sequential(
            nn.Linear(
                query_dim,
                32,
            ),
            nn.SiLU(),
        )

        self.out = nn.Sequential(
            nn.Linear(
                64,
                64,
            ),
            nn.SiLU(),
            nn.Linear(
                64,
                3,
            ),
        )

    def forward(
        self,
        evidence,
        query,
    ):
        spatial = (
            self.conv(
                evidence
            )
            .flatten(1)
        )

        qh = self.query(
            query
        )

        return (
            torch.tanh(
                self.out(
                    torch.cat(
                        [
                            spatial,
                            qh,
                        ],
                        dim=-1,
                    )
                )
            )
            * 0.75
        )


def train_center_probe(
    name,
    probe_cls,
    evidence,
    query_embeddings,
    anchors,
    gt_centers,
):
    torch.manual_seed(SEED)

    probe = probe_cls(
        evidence.shape[1],
        query_embeddings.shape[-1],
    ).to(device)

    optimizer = torch.optim.AdamW(
        probe.parameters(),
        lr=LOCAL_HEAD_LR,
        weight_decay=1e-4,
    )

    target_delta = (
        gt_centers
        - anchors
    )

    best_mean = float(
        "inf"
    )
    best_state = None

    for step in range(
        LOCAL_CENTER_STEPS + 1
    ):
        probe.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        delta = probe(
            evidence,
            query_embeddings,
        )

        loss = F.smooth_l1_loss(
            delta,
            target_delta,
            beta=0.05,
        )

        if step < LOCAL_CENTER_STEPS:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                probe.parameters(),
                1.0,
            )

            optimizer.step()

        if (
            step == 0
            or step % 50 == 0
            or step
            == LOCAL_CENTER_STEPS
        ):
            probe.eval()

            with torch.no_grad():
                refined = (
                    anchors
                    + probe(
                        evidence,
                        query_embeddings,
                    )
                )

                error = (
                    torch.linalg.vector_norm(
                        refined
                        - gt_centers,
                        dim=-1,
                    )
                )

                mean_error = float(
                    error.mean()
                )

            log(
                f"C {name} step={step} | "
                f"mean={mean_error:.4f} dref | "
                f"max={float(error.max()):.4f}"
            )

            if mean_error < best_mean:
                best_mean = mean_error
                best_state = copy.deepcopy(
                    probe.state_dict()
                )

    probe.load_state_dict(
        best_state
    )

    probe.eval()

    with torch.no_grad():
        refined = (
            anchors
            + probe(
                evidence,
                query_embeddings,
            )
        )

        error = (
            torch.linalg.vector_norm(
                refined
                - gt_centers,
                dim=-1,
            )
        )

    result = {
        "name": name,
        "mean_error_dref": float(
            error.mean()
        ),
        "max_error_dref": float(
            error.max()
        ),
        "mean_error_um": float(
            error.mean()
            * dref_um
        ),
    }

    refined_cpu = (
        refined.detach()
        .cpu()
    )

    error_cpu = (
        error.detach()
        .cpu()
    )

    del probe, optimizer
    cleanup()

    return (
        result,
        refined_cpu,
        error_cpu,
    )


center_probe_results = []
center_probe_refs = {}

if RUN_LOCAL_CENTER_EXPERIMENT:
    try:
        anchors = (
            selected_initial_refs
            .to(device)
            .float()
        )

        gt_centers_device = (
            selected_gt_centers
            .to(device)
            .float()
        )

        q0_embeddings = (
            baseline_intermediate[
                "q0"
            ].embeddings[
                0,
                selected_query_slots
                .to(device),
            ]
            .detach()
            .float()
        )

        evidence_parts = (
            build_explicit_evidence(
                anchors
            )
        )

        full_evidence = torch.cat(
            [
                evidence_parts[
                    "d0"
                ],
                evidence_parts[
                    "inputs"
                ],
                evidence_parts[
                    "dense"
                ],
                evidence_parts[
                    "rel"
                ],
            ],
            dim=1,
        )

        query_only_dummy = (
            evidence_parts[
                "rel"
            ][:, :1]
        )

        # 1. Query-only MLP.
        class QueryOnlyWrapper(
            QueryOnlyCenterProbe
        ):
            def __init__(
                self,
                in_channels,
                query_dim,
            ):
                super().__init__(
                    query_dim
                )

        experiments = [
            (
                "query_only",
                QueryOnlyWrapper,
                query_only_dummy,
            ),
            (
                "local_tensor_only",
                TensorOnlyCenterProbe,
                full_evidence,
            ),
            (
                "local_tensor_plus_query",
                TensorQueryCenterProbe,
                full_evidence,
            ),
        ]

        # Deterministic non-identity permutation.
        permutation = torch.roll(
            torch.arange(
                9,
                device=device,
            ),
            shifts=1,
        )

        experiments.append(
            (
                "shuffled_tensor_plus_query",
                TensorQueryCenterProbe,
                full_evidence[
                    permutation
                ],
            )
        )

        for (
            name,
            probe_cls,
            evidence,
        ) in experiments:
            result, refs, errors = (
                train_center_probe(
                    name,
                    probe_cls,
                    evidence,
                    q0_embeddings,
                    anchors,
                    gt_centers_device,
                )
            )

            center_probe_results.append(
                result
            )

            center_probe_refs[
                name
            ] = refs

            pd.DataFrame(
                {
                    "query_slot": (
                        selected_query_slots
                        .numpy()
                    ),
                    "gt_id": (
                        selected_gt_ids
                    ),
                    "error_dref": (
                        errors.numpy()
                    ),
                }
            ).to_csv(
                RUN_DIR
                / f"C_{name}_per_cell.csv",
                index=False,
            )

        center_probe_df = pd.DataFrame(
            center_probe_results
        )

        display(
            center_probe_df
        )

        center_probe_df.to_csv(
            RUN_DIR
            / "C_center_information_controls.csv",
            index=False,
        )

        record_result(
            "C_center_information",
            rows=(
                center_probe_df
                .to_dict("records")
            ),
        )

    except Exception as exc:
        record_error(
            "C_center_information",
            exc,
        )

        center_probe_df = pd.DataFrame(
            center_probe_results
        )

        cleanup()

else:
    center_probe_df = pd.DataFrame()


## Experiment D — Corrected coarse-mask target and mask-lattice test on all 9 cells

The current dot-product mask form is kept fixed:

\[
M_i(x)=e_i^\top F(x)
\]

Only copied versions of the final mask embedding/projection heads are trained.

We compare:

```text
2048 + nearest multiclass-label target
2048 + occupancy-preserving per-instance target
8192 + occupancy-preserving target
16384 + occupancy-preserving target
```

The fixed nine query↔GT pairs from Section 3 are used for every variant.

Training uses short convergence/early stopping rather than a fixed 60-step budget.


In [ ]:
def cap_feature_tokens_local(
    feature,
    spacing_um,
    max_tokens,
):
    z, y, x = (
        feature.shape[-3:]
    )

    total = z * y * x

    if total <= max_tokens:
        return (
            feature,
            spacing_um,
        )

    scale = (
        total / max_tokens
    ) ** (1.0 / 3.0)

    target_shape = tuple(
        max(
            1,
            int(round(v / scale)),
        )
        for v in (
            z,
            y,
            x,
        )
    )

    while (
        np.prod(
            target_shape
        )
        > max_tokens
    ):
        axis = max(
            range(3),
            key=lambda i: (
                target_shape[i]
            ),
        )

        target_shape = tuple(
            (
                value - 1
                if (
                    i == axis
                    and value > 1
                )
                else value
            )
            for i, value
            in enumerate(
                target_shape
            )
        )

    pooled = (
        F.adaptive_avg_pool3d(
            feature,
            target_shape,
        )
    )

    ratio = torch.tensor(
        [
            (
                (size - 1)
                / (out - 1)
                if out > 1
                else 1.0
            )
            for size, out
            in zip(
                (z, y, x),
                target_shape,
            )
        ],
        device=feature.device,
        dtype=spacing_um.dtype,
    )

    return (
        pooled,
        spacing_um
        * ratio[None],
    )


def occupancy_targets(
    shape,
    gt_ids_ordered,
):
    masks = []

    for gt_id in gt_ids_ordered:
        native = torch.from_numpy(
            (
                gt_labels_native
                == int(gt_id)
            ).astype(
                np.float32
            )
        )[
            None,
            None,
        ]

        pooled = (
            F.adaptive_max_pool3d(
                native,
                shape,
            )[0, 0]
        )

        masks.append(pooled)

    return torch.stack(
        masks,
        dim=0,
    ).to(device)


def nearest_targets(
    shape,
):
    ordered_target_indices = (
        source9_gt_indices[
            selected_gt_rows
        ]
    )

    return target_masks_at_shape(
        target,
        shape,
        device,
        target_indices=(
            ordered_target_indices
        ),
    )


def local_mask_metrics(
    logits,
    gt,
    support,
):
    probability = (
        logits.float()
        .sigmoid()
    )

    p_local = (
        probability
        * support.float()
    )

    intersection = (
        p_local
        * gt.float()
    ).flatten(1).sum(-1)

    soft_dice = (
        2 * intersection
        + 1e-6
    ) / (
        p_local
        .flatten(1)
        .sum(-1)
        + gt.float()
        .flatten(1)
        .sum(-1)
        + 1e-6
    )

    hard = (
        (probability >= 0.5)
        & support.bool()
    )

    gt_bool = gt.bool()

    tp = (
        hard
        & gt_bool
    ).flatten(1).sum(-1).float()

    fp = (
        hard
        & ~gt_bool
    ).flatten(1).sum(-1).float()

    fn = (
        ~hard
        & gt_bool
    ).flatten(1).sum(-1).float()

    hard_dice = (
        2 * tp
        + 1e-6
    ) / (
        2 * tp
        + fp
        + fn
        + 1e-6
    )

    return (
        soft_dice,
        hard_dice,
    )


class DotMaskProbe(nn.Module):
    def __init__(
        self,
        mask_embed,
        mask_feature_proj,
    ):
        super().__init__()

        self.mask_embed = (
            copy.deepcopy(
                mask_embed
            )
        )

        self.mask_feature_proj = (
            copy.deepcopy(
                mask_feature_proj
            )
        )

    def forward(
        self,
        query_embeddings,
        spatial_feature,
    ):
        mask_embedding = (
            self.mask_embed(
                query_embeddings
            )
        )

        mask_feature = (
            self.mask_feature_proj(
                spatial_feature
            )[0]
        )

        logits = torch.einsum(
            "qc,cv->qv",
            mask_embedding,
            mask_feature.flatten(1),
        )

        return logits.reshape(
            len(query_embeddings),
            *mask_feature.shape[-3:],
        )


def train_dot_mask_probe(
    token_cap,
    target_kind,
):
    d1 = (
        baseline_intermediate[
            "d1"
        ].detach()
    )

    d1_spacing = (
        baseline_intermediate[
            "pyramid"
        ].spacings_um[1]
    )

    spatial, spacing = (
        cap_feature_tokens_local(
            d1,
            d1_spacing,
            token_cap,
        )
    )

    shape = tuple(
        int(v)
        for v
        in spatial.shape[-3:]
    )

    if target_kind == "nearest":
        gt = nearest_targets(
            shape
        )
    elif target_kind == "occupancy":
        gt = occupancy_targets(
            shape,
            selected_gt_ids,
        )
    else:
        raise ValueError(
            target_kind
        )

    gt = gt.float()

    query_embeddings = (
        baseline_outputs
        .query_embeddings[
            0,
            selected_query_slots
            .to(device),
        ]
        .detach()
        .float()
    )

    gt_centers = (
        selected_gt_centers
        .to(device)
        .float()
    )

    support = (
        build_local_support_masks(
            gt,
            gt_centers,
            spacing[0],
            b["dref_um"][0],
            float(
                cfg.losses
                .mask_supervision_radius_dref
            ),
        )
    )

    probe = DotMaskProbe(
        model.query_decoder
        .layers[-1]
        .mask_embed,
        model.query_decoder
        .mask_feature_proj[-1],
    ).to(device).float()

    optimizer = torch.optim.AdamW(
        probe.parameters(),
        lr=DOT_MASK_LR,
        weight_decay=1e-4,
    )

    spatial = (
        spatial.detach().float()
    )

    best_dice = -1.0
    best_state = None
    best_step = 0
    evaluations_without_gain = 0

    for step in range(
        DOT_MASK_MAX_STEPS + 1
    ):
        probe.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = probe(
            query_embeddings,
            spatial,
        )

        probability = (
            logits.sigmoid()
        )

        p_local = (
            probability
            * support.float()
        )

        intersection = (
            p_local
            * gt
        ).flatten(1).sum(-1)

        dice = (
            2 * intersection
            + 1e-6
        ) / (
            p_local
            .flatten(1)
            .sum(-1)
            + gt
            .flatten(1)
            .sum(-1)
            + 1e-6
        )

        bce_terms = []

        for row in range(
            len(gt)
        ):
            local = support[
                row
            ]

            bce_terms.append(
                F.binary_cross_entropy_with_logits(
                    logits[row][local],
                    gt[row][local],
                )
            )

        loss = (
            1
            - dice.mean()
            + 0.5
            * torch.stack(
                bce_terms
            ).mean()
        )

        if step < DOT_MASK_MAX_STEPS:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                probe.parameters(),
                1.0,
            )

            optimizer.step()

        should_eval = (
            step == 0
            or step
            % DOT_MASK_EVAL_EVERY
            == 0
            or step
            == DOT_MASK_MAX_STEPS
        )

        if should_eval:
            probe.eval()
            with torch.no_grad():
                eval_logits = probe(
                    query_embeddings,
                    spatial,
                )
                eval_soft, _ = local_mask_metrics(
                    eval_logits,
                    gt,
                    support,
                )
                mean_dice = float(
                    eval_soft.mean()
                    .detach()
                    .cpu()
                )

            log(
                f"D cap={token_cap} "
                f"target={target_kind} "
                f"step={step} "
                f"Dice={mean_dice:.4f}"
            )

            if (
                mean_dice
                > best_dice
                + 1e-4
            ):
                best_dice = mean_dice
                best_step = step
                best_state = (
                    copy.deepcopy(
                        probe.state_dict()
                    )
                )
                evaluations_without_gain = 0

            else:
                evaluations_without_gain += 1

            if (
                step
                >= DOT_MASK_MIN_STEPS
                and evaluations_without_gain
                >= DOT_MASK_PATIENCE_EVALS
            ):
                log(
                    f"D early stop cap={token_cap} "
                    f"target={target_kind} "
                    f"at step={step}"
                )
                break

    if best_state is None:
        raise RuntimeError(
            "Dot-mask probe never produced a best state."
        )

    probe.load_state_dict(
        best_state
    )

    probe.eval()

    with torch.no_grad():
        logits = probe(
            query_embeddings,
            spatial,
        )

        soft_dice, hard_dice = (
            local_mask_metrics(
                logits,
                gt,
                support,
            )
        )

    positive_counts = (
        gt.flatten(1)
        .sum(-1)
        .detach()
        .cpu()
        .numpy()
    )

    result = {
        "token_cap": int(
            token_cap
        ),
        "target_kind": (
            target_kind
        ),
        "shape": str(
            shape
        ),
        "spatial_positions": int(
            np.prod(shape)
        ),
        "zero_gt_cells": int(
            (
                positive_counts
                == 0
            ).sum()
        ),
        "mean_gt_positive_voxels": float(
            positive_counts.mean()
        ),
        "min_gt_positive_voxels": float(
            positive_counts.min()
        ),
        "soft_dice_mean": float(
            soft_dice.mean()
            .cpu()
        ),
        "soft_dice_min": float(
            soft_dice.min()
            .cpu()
        ),
        "hard_dice_mean": float(
            hard_dice.mean()
            .cpu()
        ),
        "best_step": int(
            best_step
        ),
    }

    detail = pd.DataFrame(
        {
            "query_slot": (
                selected_query_slots
                .numpy()
            ),
            "gt_id": (
                selected_gt_ids
            ),
            "positive_target_voxels": (
                positive_counts
            ),
            "soft_dice": (
                soft_dice
                .cpu()
                .numpy()
            ),
            "hard_dice": (
                hard_dice
                .cpu()
                .numpy()
            ),
        }
    )

    del (
        probe,
        optimizer,
        logits,
        gt,
    )

    cleanup()

    return result, detail


coarse_probe_results = []

if RUN_COARSE_MASK_PROBE:
    try:
        probe_specs = [
            (
                2048,
                "nearest",
            ),
            (
                2048,
                "occupancy",
            ),
            (
                8192,
                "occupancy",
            ),
            (
                16384,
                "occupancy",
            ),
        ]

        for (
            token_cap,
            target_kind,
        ) in probe_specs:
            result, detail = (
                train_dot_mask_probe(
                    token_cap,
                    target_kind,
                )
            )

            coarse_probe_results.append(
                result
            )

            detail.to_csv(
                RUN_DIR
                / (
                    f"D_{token_cap}_"
                    f"{target_kind}_"
                    "per_cell.csv"
                ),
                index=False,
            )

            record_result(
                "D_coarse_dot_mask_probe",
                **result,
            )

        coarse_probe_df = pd.DataFrame(
            coarse_probe_results
        )

        display(
            coarse_probe_df
        )

        coarse_probe_df.to_csv(
            RUN_DIR
            / "D_coarse_dot_mask_probe.csv",
            index=False,
        )

    except Exception as exc:
        record_error(
            "D_coarse_dot_mask_probe",
            exc,
        )

        coarse_probe_df = pd.DataFrame(
            coarse_probe_results
        )

        cleanup()

else:
    coarse_probe_df = pd.DataFrame()


## Experiment E — Corrected local-mask information test with anti-memorization controls

All nine fixed query↔GT pairs are used.

Full-fit variants:

```text
xyz_common_query
query_xyz
d0_query
d0_raw_query
explicit_all_query
explicit_all_common_query
explicit_all_shuffled
oracle_center_explicit_all_query
```

The critical controls are:

- `query_xyz`: can the query alone memorize the masks?
- `explicit_all_common_query`: can local geometry work even when all query identities are identical?
- `explicit_all_shuffled`: what happens if the spatial evidence belongs to the wrong cell?
- oracle centers: how much does center error matter once local geometry is available?

A leave-one-cell-out head experiment then measures whether the local mask head can use spatial evidence on a cell whose mask was **not used to train that head**.


In [ ]:
class LocalEvidenceMaskHead(nn.Module):
    def __init__(
        self,
        in_channels,
        query_dim,
        hidden=32,
    ):
        super().__init__()

        self.evidence = nn.Sequential(
            nn.Conv3d(
                in_channels,
                hidden,
                3,
                padding=1,
            ),
            nn.GroupNorm(
                4,
                hidden,
            ),
            nn.SiLU(),
            nn.Conv3d(
                hidden,
                hidden,
                3,
                padding=1,
            ),
            nn.GroupNorm(
                4,
                hidden,
            ),
            nn.SiLU(),
        )

        self.query = nn.Sequential(
            nn.Linear(
                query_dim,
                hidden,
            ),
            nn.SiLU(),
            nn.Linear(
                hidden,
                hidden,
            ),
        )

        self.fuse = nn.Sequential(
            nn.Conv3d(
                2 * hidden,
                hidden,
                1,
            ),
            nn.SiLU(),
            nn.Conv3d(
                hidden,
                1,
                1,
            ),
        )

    def forward(
        self,
        evidence,
        query,
    ):
        feature = self.evidence(
            evidence
        )

        query_feature = (
            self.query(query)[
                :,
                :,
                None,
                None,
                None,
            ]
            .expand_as(feature)
        )

        return (
            self.fuse(
                torch.cat(
                    [
                        feature,
                        query_feature,
                    ],
                    dim=1,
                )
            )[:, 0]
        )


def soft_hard_dice_cubes(
    logits,
    gt,
):
    probability = logits.sigmoid()

    intersection = (
        probability
        * gt
    ).flatten(1).sum(-1)

    soft = (
        2 * intersection
        + 1e-6
    ) / (
        probability
        .flatten(1)
        .sum(-1)
        + gt
        .flatten(1)
        .sum(-1)
        + 1e-6
    )

    hard = probability >= 0.5
    gt_bool = gt.bool()

    tp = (
        hard
        & gt_bool
    ).flatten(1).sum(-1).float()

    fp = (
        hard
        & ~gt_bool
    ).flatten(1).sum(-1).float()

    fn = (
        ~hard
        & gt_bool
    ).flatten(1).sum(-1).float()

    hard_dice = (
        2 * tp
        + 1e-6
    ) / (
        2 * tp
        + fp
        + fn
        + 1e-6
    )

    return soft, hard_dice


def train_local_mask_full_fit(
    name,
    evidence,
    query_embeddings,
    gt,
):
    torch.manual_seed(SEED)

    head = LocalEvidenceMaskHead(
        evidence.shape[1],
        query_embeddings.shape[-1],
    ).to(device)

    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=LOCAL_HEAD_LR,
        weight_decay=1e-4,
    )

    best_dice = -1.0
    best_state = None

    for step in range(
        LOCAL_MASK_STEPS + 1
    ):
        head.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = head(
            evidence,
            query_embeddings,
        )

        soft, _ = (
            soft_hard_dice_cubes(
                logits,
                gt,
            )
        )

        loss = (
            1
            - soft.mean()
            + 0.5
            * F.binary_cross_entropy_with_logits(
                logits,
                gt,
            )
        )

        if step < LOCAL_MASK_STEPS:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                head.parameters(),
                1.0,
            )

            optimizer.step()

        if (
            step == 0
            or step % 30 == 0
            or step
            == LOCAL_MASK_STEPS
        ):
            head.eval()
            with torch.no_grad():
                eval_logits = head(
                    evidence,
                    query_embeddings,
                )
                eval_soft, _ = soft_hard_dice_cubes(
                    eval_logits,
                    gt,
                )
                mean_dice = float(
                    eval_soft.mean()
                    .detach()
                    .cpu()
                )

            log(
                f"E {name} step={step} | "
                f"softDice={mean_dice:.4f}"
            )

            if mean_dice > best_dice:
                best_dice = (
                    mean_dice
                )
                best_state = (
                    copy.deepcopy(
                        head.state_dict()
                    )
                )

    if best_state is None:
        raise RuntimeError(
            f"{name}: no best state"
        )

    head.load_state_dict(
        best_state
    )

    head.eval()

    with torch.no_grad():
        logits = head(
            evidence,
            query_embeddings,
        )

        soft, hard = (
            soft_hard_dice_cubes(
                logits,
                gt,
            )
        )

        probability = (
            logits.sigmoid()
        )

    result = {
        "name": name,
        "soft_dice_mean": float(
            soft.mean().cpu()
        ),
        "soft_dice_min": float(
            soft.min().cpu()
        ),
        "hard_dice_mean": float(
            hard.mean().cpu()
        ),
        "evidence_channels": int(
            evidence.shape[1]
        ),
    }

    detail = pd.DataFrame(
        {
            "query_slot": (
                selected_query_slots
                .numpy()
            ),
            "gt_id": (
                selected_gt_ids
            ),
            "soft_dice": (
                soft.cpu()
                .numpy()
            ),
            "hard_dice": (
                hard.cpu()
                .numpy()
            ),
        }
    )

    prediction = (
        probability
        .detach()
        .cpu()
        .to(torch.float16)
        .numpy()
    )

    del (
        head,
        optimizer,
        logits,
    )

    cleanup()

    return (
        result,
        detail,
        prediction,
    )


local_mask_results = []
local_mask_predictions = {}
loo_mask_df = pd.DataFrame()

if RUN_LOCAL_MASK_EXPERIMENT:
    try:
        predicted_refs = (
            selected_final_refs
            .to(device)
            .float()
        )

        oracle_refs = (
            selected_gt_centers
            .to(device)
            .float()
        )

        query_embeddings = (
            baseline_outputs
            .query_embeddings[
                0,
                selected_query_slots
                .to(device),
            ]
            .detach()
            .float()
        )

        common_query = (
            query_embeddings
            .mean(
                dim=0,
                keepdim=True,
            )
            .expand_as(
                query_embeddings
            )
        )

        predicted_parts = (
            build_explicit_evidence(
                predicted_refs
            )
        )

        oracle_parts = (
            build_explicit_evidence(
                oracle_refs
            )
        )

        gt_predicted = (
            sample_gt_cubes(
                selected_gt_ids,
                predicted_refs,
            ).float()
        )

        gt_oracle = (
            sample_gt_cubes(
                selected_gt_ids,
                oracle_refs,
            ).float()
        )

        predicted_evidence = {
            "xyz_common_query": (
                predicted_parts[
                    "rel"
                ]
            ),
            "query_xyz": (
                predicted_parts[
                    "rel"
                ]
            ),
            "d0_query": torch.cat(
                [
                    predicted_parts[
                        "d0"
                    ],
                    predicted_parts[
                        "rel"
                    ],
                ],
                dim=1,
            ),
            "d0_raw_query": torch.cat(
                [
                    predicted_parts[
                        "d0"
                    ],
                    predicted_parts[
                        "inputs"
                    ][:, 0:1],
                    predicted_parts[
                        "rel"
                    ],
                ],
                dim=1,
            ),
            "explicit_all_query": torch.cat(
                [
                    predicted_parts[
                        "d0"
                    ],
                    predicted_parts[
                        "inputs"
                    ],
                    predicted_parts[
                        "dense"
                    ],
                    predicted_parts[
                        "rel"
                    ],
                ],
                dim=1,
            ),
        }

        full_explicit = (
            predicted_evidence[
                "explicit_all_query"
            ]
        )

        permutation = torch.roll(
            torch.arange(
                9,
                device=device,
            ),
            shifts=1,
        )

        variant_specs = [
            (
                "xyz_common_query",
                predicted_evidence[
                    "xyz_common_query"
                ],
                common_query,
                gt_predicted,
            ),
            (
                "query_xyz",
                predicted_evidence[
                    "query_xyz"
                ],
                query_embeddings,
                gt_predicted,
            ),
            (
                "d0_query",
                predicted_evidence[
                    "d0_query"
                ],
                query_embeddings,
                gt_predicted,
            ),
            (
                "d0_raw_query",
                predicted_evidence[
                    "d0_raw_query"
                ],
                query_embeddings,
                gt_predicted,
            ),
            (
                "explicit_all_query",
                full_explicit,
                query_embeddings,
                gt_predicted,
            ),
            (
                "explicit_all_common_query",
                full_explicit,
                common_query,
                gt_predicted,
            ),
            (
                "explicit_all_shuffled",
                full_explicit[
                    permutation
                ],
                query_embeddings,
                gt_predicted,
            ),
            (
                "oracle_center_explicit_all_query",
                torch.cat(
                    [
                        oracle_parts[
                            "d0"
                        ],
                        oracle_parts[
                            "inputs"
                        ],
                        oracle_parts[
                            "dense"
                        ],
                        oracle_parts[
                            "rel"
                        ],
                    ],
                    dim=1,
                ),
                query_embeddings,
                gt_oracle,
            ),
        ]

        for (
            name,
            evidence,
            q_input,
            gt_input,
        ) in variant_specs:
            (
                result,
                detail,
                prediction,
            ) = train_local_mask_full_fit(
                name,
                evidence,
                q_input,
                gt_input,
            )

            local_mask_results.append(
                result
            )

            local_mask_predictions[
                name
            ] = prediction

            detail.to_csv(
                RUN_DIR
                / f"E_{name}_per_cell.csv",
                index=False,
            )

            record_result(
                "E_local_mask_full_fit",
                **result,
            )

        local_mask_df = pd.DataFrame(
            local_mask_results
        )

        display(
            local_mask_df
        )

        local_mask_df.to_csv(
            RUN_DIR
            / "E_local_mask_information_full_fit.csv",
            index=False,
        )

        np.savez_compressed(
            RUN_DIR
            / "E_local_mask_predictions_small.npz",
            gt_ids=selected_gt_ids,
            query_slots=(
                selected_query_slots
                .numpy()
            ),
            predicted_centers=(
                selected_final_refs
                .numpy()
            ),
            gt_centers=(
                selected_gt_centers
                .numpy()
            ),
            **local_mask_predictions,
        )

    except Exception as exc:
        record_error(
            "E_local_mask_full_fit",
            exc,
        )

        local_mask_df = pd.DataFrame(
            local_mask_results
        )

        cleanup()

else:
    local_mask_df = pd.DataFrame()


### Experiment E2 — Leave-one-cell-out local-mask head validation

This is the anti-memorization check.

For each of the 9 cells:

```text
train temporary mask head on 8 cells
evaluate on the held-out 9th cell
```

The STIR-Net backbone/query checkpoint remains frozen.

Compared variants:

- `query_xyz`
- `explicit_all_query`
- `explicit_all_common_query`
- `explicit_all_shuffled`

If correctly aligned explicit spatial evidence performs substantially better than query+XYZ and shuffled evidence on held-out cells, then the local image/geometry tensor is genuinely informative rather than only supporting memorization.


In [ ]:
def train_one_loo_head(
    evidence,
    query_input,
    gt,
    train_rows,
    test_row,
):
    torch.manual_seed(SEED)

    head = LocalEvidenceMaskHead(
        evidence.shape[1],
        query_input.shape[-1],
    ).to(device)

    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=LOCAL_HEAD_LR,
        weight_decay=1e-4,
    )

    for step in range(
        LOO_MASK_STEPS
    ):
        head.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = head(
            evidence[
                train_rows
            ],
            query_input[
                train_rows
            ],
        )

        gt_train = gt[
            train_rows
        ]

        soft, _ = (
            soft_hard_dice_cubes(
                logits,
                gt_train,
            )
        )

        loss = (
            1
            - soft.mean()
            + 0.5
            * F.binary_cross_entropy_with_logits(
                logits,
                gt_train,
            )
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            head.parameters(),
            1.0,
        )

        optimizer.step()

    head.eval()

    with torch.no_grad():
        test_logits = head(
            evidence[
                test_row : test_row + 1
            ],
            query_input[
                test_row : test_row + 1
            ],
        )

        test_soft, test_hard = (
            soft_hard_dice_cubes(
                test_logits,
                gt[
                    test_row
                    : test_row + 1
                ],
            )
        )

    result = (
        float(
            test_soft[0].cpu()
        ),
        float(
            test_hard[0].cpu()
        ),
    )

    del (
        head,
        optimizer,
        logits,
        test_logits,
    )

    cleanup()

    return result


if (
    RUN_LOCAL_MASK_EXPERIMENT
    and RUN_LOO_MASK_VALIDATION
    and len(local_mask_df)
):
    try:
        # Reuse correctly aligned predicted-center tensors from E.
        correct_explicit = (
            predicted_evidence[
                "explicit_all_query"
            ]
        )

        xyz_evidence = (
            predicted_evidence[
                "query_xyz"
            ]
        )

        gt = gt_predicted

        loo_rows = []

        for held_out in range(9):
            train_rows = torch.tensor(
                [
                    i
                    for i in range(9)
                    if i != held_out
                ],
                device=device,
                dtype=torch.long,
            )

            train_common = (
                query_embeddings[
                    train_rows
                ]
                .mean(
                    dim=0,
                    keepdim=True,
                )
            )

            common_q_fold = (
                train_common.expand(
                    9,
                    -1,
                )
            )

            # Fold-specific shuffled control:
            # training evidence is rotated only within training rows,
            # and the held-out sample receives one training cell's evidence.
            shuffled_explicit = (
                correct_explicit.clone()
            )

            rotated_train = torch.roll(
                train_rows,
                shifts=1,
            )

            shuffled_explicit[
                train_rows
            ] = correct_explicit[
                rotated_train
            ]

            shuffled_explicit[
                held_out
            ] = correct_explicit[
                train_rows[0]
            ]

            fold_specs = [
                (
                    "query_xyz",
                    xyz_evidence,
                    query_embeddings,
                ),
                (
                    "explicit_all_query",
                    correct_explicit,
                    query_embeddings,
                ),
                (
                    "explicit_all_common_query",
                    correct_explicit,
                    common_q_fold,
                ),
                (
                    "explicit_all_shuffled",
                    shuffled_explicit,
                    query_embeddings,
                ),
            ]

            for (
                variant,
                evidence,
                q_input,
            ) in fold_specs:
                (
                    soft_dice,
                    hard_dice,
                ) = train_one_loo_head(
                    evidence,
                    q_input,
                    gt,
                    train_rows,
                    held_out,
                )

                loo_rows.append(
                    {
                        "variant": variant,
                        "held_out_row": (
                            held_out
                        ),
                        "gt_id": int(
                            selected_gt_ids[
                                held_out
                            ]
                        ),
                        "soft_dice": (
                            soft_dice
                        ),
                        "hard_dice": (
                            hard_dice
                        ),
                    }
                )

                log(
                    f"E2 heldout GT={selected_gt_ids[held_out]} "
                    f"variant={variant} "
                    f"softDice={soft_dice:.4f}"
                )

        loo_mask_per_cell_df = pd.DataFrame(
            loo_rows
        )

        loo_mask_df = (
            loo_mask_per_cell_df
            .groupby(
                "variant"
            )
            .agg(
                soft_dice_mean=(
                    "soft_dice",
                    "mean",
                ),
                soft_dice_median=(
                    "soft_dice",
                    "median",
                ),
                soft_dice_min=(
                    "soft_dice",
                    "min",
                ),
                hard_dice_mean=(
                    "hard_dice",
                    "mean",
                ),
            )
            .reset_index()
        )

        display(
            loo_mask_df
        )

        loo_mask_per_cell_df.to_csv(
            RUN_DIR
            / "E2_loo_mask_per_cell.csv",
            index=False,
        )

        loo_mask_df.to_csv(
            RUN_DIR
            / "E2_loo_mask_summary.csv",
            index=False,
        )

        record_result(
            "E2_loo_mask_validation",
            rows=(
                loo_mask_df
                .to_dict("records")
            ),
        )

    except Exception as exc:
        record_error(
            "E2_loo_mask_validation",
            exc,
        )

        loo_mask_df = pd.DataFrame()

        cleanup()


## Experiment F — Support-radius geometry on all nine fixed cells

This phase asks how large a radial render/support region actually needs to be.

For each real cell it measures GT-voxel coverage around:

- the GT center;
- the current matched predicted center.

It also reports the spherical support volume relative to the real GT-cell volume.


In [ ]:
support_rows = []
support_summary = pd.DataFrame()

if RUN_SUPPORT_AUDIT:
    try:
        full_shape = np.asarray(
            gt_labels_native.shape,
            dtype=np.float64,
        )

        extent_um = (
            full_shape - 1
        ) * spacing_native

        voxel_volume_um3 = float(
            np.prod(
                spacing_native
            )
        )

        for (
            gt_id,
            gt_center_dref,
            pred_center_dref,
        ) in zip(
            selected_gt_ids,
            selected_gt_centers.numpy(),
            selected_final_refs.numpy(),
        ):
            voxels = np.argwhere(
                gt_labels_native
                == int(gt_id)
            ).astype(
                np.float64
            )

            voxel_positions_um = (
                voxels
                * spacing_native[None]
                - 0.5
                * extent_um[None]
            )

            gt_center_um = (
                gt_center_dref
                * dref_um
            )

            pred_center_um = (
                pred_center_dref
                * dref_um
            )

            distance_gt = (
                np.linalg.norm(
                    voxel_positions_um
                    - gt_center_um[None],
                    axis=1,
                )
            )

            distance_pred = (
                np.linalg.norm(
                    voxel_positions_um
                    - pred_center_um[None],
                    axis=1,
                )
            )

            gt_voxel_count = len(
                voxels
            )

            for radius_dref in (
                SUPPORT_RADII_DREF
            ):
                radius_um = (
                    radius_dref
                    * dref_um
                )

                sphere_equivalent_voxels = (
                    (
                        4.0
                        / 3.0
                    )
                    * math.pi
                    * radius_um**3
                    / voxel_volume_um3
                )

                support_rows.append(
                    {
                        "gt_id": (
                            int(gt_id)
                        ),
                        "radius_dref": (
                            float(
                                radius_dref
                            )
                        ),
                        "coverage_from_gt_center": float(
                            (
                                distance_gt
                                <= radius_um
                            ).mean()
                        ),
                        "coverage_from_pred_center": float(
                            (
                                distance_pred
                                <= radius_um
                            ).mean()
                        ),
                        "sphere_to_gt_volume_ratio": float(
                            sphere_equivalent_voxels
                            / max(
                                gt_voxel_count,
                                1,
                            )
                        ),
                        "gt_voxels": int(
                            gt_voxel_count
                        ),
                    }
                )

        support_df = pd.DataFrame(
            support_rows
        )

        support_summary = (
            support_df
            .groupby(
                "radius_dref"
            )
            .agg(
                gt_center_coverage_mean=(
                    "coverage_from_gt_center",
                    "mean",
                ),
                gt_center_coverage_min=(
                    "coverage_from_gt_center",
                    "min",
                ),
                pred_center_coverage_mean=(
                    "coverage_from_pred_center",
                    "mean",
                ),
                pred_center_coverage_min=(
                    "coverage_from_pred_center",
                    "min",
                ),
                sphere_to_gt_volume_ratio_mean=(
                    "sphere_to_gt_volume_ratio",
                    "mean",
                ),
            )
            .reset_index()
        )

        display(
            support_summary
        )

        support_df.to_csv(
            RUN_DIR
            / "F_support_per_cell.csv",
            index=False,
        )

        support_summary.to_csv(
            RUN_DIR
            / "F_support_summary.csv",
            index=False,
        )

        record_result(
            "F_support_geometry",
            rows=(
                support_summary
                .to_dict("records")
            ),
        )

    except Exception as exc:
        record_error(
            "F_support_geometry",
            exc,
        )

        support_df = pd.DataFrame()
        support_summary = pd.DataFrame()

        cleanup()


## 6. Causal diagnosis

The automatic summary is deliberately conservative.

It distinguishes:

- **confirmed implementation/target defect**,
- **evidence-supported architectural change**,
- **proposal that remains unproven**.

The leave-one-cell-out mask controls take priority over full-fit mask Dice when deciding whether explicit spatial evidence is genuinely useful.


In [ ]:
diagnosis = {
    "generated_at": now_text(),
    "checkpoint": str(
        BEST_SPATIAL_QUERY_CHECKPOINT
    ),
    "selection_invariant": {
        "pool_size": int(
            len(pool_slots)
        ),
        "selected_query_count": int(
            len(selected_query_slots)
        ),
        "selected_unique_gt_count": int(
            len(
                np.unique(
                    selected_gt_ids
                )
            )
        ),
        "selected_gt_ids": (
            selected_gt_ids.tolist()
        ),
        "off_mask_selected_queries": int(
            (
                selection_audit[
                    "source_instance_id"
                ]
                < 0
            ).sum()
        ),
    },
    "confirmed_findings": [],
    "supported_architecture_changes": [],
    "not_yet_proven": [],
}


# ---------------------------------------------------------------
# Selection invariant
# ---------------------------------------------------------------

if (
    len(selected_query_slots) != 9
    or len(
        np.unique(
            selected_gt_ids
        )
    )
    != 9
):
    raise AssertionError(
        "Notebook 28B invariant failed: "
        "experiments are not using 9 query↔GT pairs."
    )


# ---------------------------------------------------------------
# A — initial/final centers
# ---------------------------------------------------------------

paired_initial_mean = float(
    paired_initial_error.mean()
)

paired_final_mean = float(
    paired_final_error.mean()
)

if paired_final_mean > (
    paired_initial_mean
    + 0.10
):
    diagnosis[
        "confirmed_findings"
    ].append(
        "For the same nine GT-aware paired proposal queries, "
        "the current query-decoder center refinement makes mean center error worse."
    )
elif paired_final_mean < (
    paired_initial_mean
    - 0.10
):
    diagnosis[
        "confirmed_findings"
    ].append(
        "For the same nine paired queries, current query-decoder center refinement improves center error."
    )
else:
    diagnosis[
        "confirmed_findings"
    ].append(
        "Initial and final center errors are similar; center inaccuracies begin at proposal localization and are not mainly created by query refinement."
    )


# ---------------------------------------------------------------
# B — relational proposal reasoning
# ---------------------------------------------------------------

if len(proposal_set_df):
    independent_rows = (
        proposal_set_df[
            proposal_set_df[
                "model"
            ]
            == "independent_refiner"
        ]
    )

    set_rows = (
        proposal_set_df[
            proposal_set_df[
                "model"
            ]
            == "set_refiner"
        ]
    )

    if (
        len(independent_rows)
        and len(set_rows)
    ):
        independent = (
            independent_rows.iloc[0]
        )

        set_result = (
            set_rows.iloc[0]
        )

        set_better = (
            (
                set_result[
                    "recall_0p5"
                ]
                > independent[
                    "recall_0p5"
                ]
                + 0.05
            )
            or (
                (
                    set_result[
                        "duplicate_nearest_gt_count"
                    ]
                    < independent[
                        "duplicate_nearest_gt_count"
                    ]
                )
                and (
                    set_result[
                        "hungarian_mean_dref"
                    ]
                    < independent[
                        "hungarian_mean_dref"
                    ]
                )
            )
        )

        if set_better:
            diagnosis[
                "supported_architecture_changes"
            ].append(
                "Candidate-to-candidate proposal-set reasoning outperforms independent refinement; "
                "a learned proposal-set refinement stage is supported."
            )
        else:
            diagnosis[
                "not_yet_proven"
            ].append(
                "Proposal-set self-attention did not clearly outperform an independent candidate refiner; "
                "do not add it solely for deduplication based on this scene."
            )
else:
    diagnosis[
        "not_yet_proven"
    ].append(
        "Proposal-set reasoning experiment did not complete."
    )


# ---------------------------------------------------------------
# C — center information
# ---------------------------------------------------------------

if len(center_probe_df):
    center_by_name = (
        center_probe_df
        .set_index(
            "name"
        )
    )

    required_center = {
        "query_only",
        "local_tensor_only",
        "local_tensor_plus_query",
        "shuffled_tensor_plus_query",
    }

    if required_center.issubset(
        set(
            center_by_name.index
        )
    ):
        query_only = float(
            center_by_name.loc[
                "query_only",
                "mean_error_dref",
            ]
        )

        tensor_only = float(
            center_by_name.loc[
                "local_tensor_only",
                "mean_error_dref",
            ]
        )

        tensor_query = float(
            center_by_name.loc[
                "local_tensor_plus_query",
                "mean_error_dref",
            ]
        )

        shuffled = float(
            center_by_name.loc[
                "shuffled_tensor_plus_query",
                "mean_error_dref",
            ]
        )

        if (
            tensor_query
            < query_only * 0.8
            and tensor_query
            < shuffled * 0.8
        ):
            diagnosis[
                "supported_architecture_changes"
            ].append(
                "Direct local 3-D evidence improves center localization beyond query-only and shuffled-evidence controls; "
                "center refinement should have direct local spatial access."
            )
        elif query_only <= (
            tensor_query
            * 1.05
        ):
            diagnosis[
                "confirmed_findings"
            ].append(
                "A new query-only center probe is already as accurate as the local-tensor probe; "
                "the existing query contains enough center information, so the current center-head/training path is the more likely issue."
            )
        elif tensor_only <= (
            tensor_query
            * 1.05
        ):
            diagnosis[
                "confirmed_findings"
            ].append(
                "Local spatial evidence alone is sufficient for accurate center correction on this scene; "
                "query identity is not required for the center correction probe."
            )


# ---------------------------------------------------------------
# D — target and resolution
# ---------------------------------------------------------------

if len(coarse_probe_df):
    nearest_2048 = (
        coarse_probe_df[
            (
                coarse_probe_df[
                    "token_cap"
                ]
                == 2048
            )
            & (
                coarse_probe_df[
                    "target_kind"
                ]
                == "nearest"
            )
        ]
    )

    occupancy_2048 = (
        coarse_probe_df[
            (
                coarse_probe_df[
                    "token_cap"
                ]
                == 2048
            )
            & (
                coarse_probe_df[
                    "target_kind"
                ]
                == "occupancy"
            )
        ]
    )

    if (
        len(nearest_2048)
        and len(
            occupancy_2048
        )
    ):
        nearest = (
            nearest_2048.iloc[0]
        )

        occupancy = (
            occupancy_2048.iloc[0]
        )

        if (
            int(
                nearest[
                    "zero_gt_cells"
                ]
            )
            > 0
            and int(
                occupancy[
                    "zero_gt_cells"
                ]
            )
            == 0
        ):
            diagnosis[
                "confirmed_findings"
            ].append(
                "Nearest-neighbor resizing of the multiclass GT label map erases real source-9 instances; "
                "per-instance occupancy-preserving coarse targets are required."
            )

        if (
            occupancy[
                "soft_dice_mean"
            ]
            > nearest[
                "soft_dice_mean"
            ]
            + 0.05
        ):
            diagnosis[
                "confirmed_findings"
            ].append(
                "At the same 2048-position mask lattice, occupancy-preserving targets materially improve dot-mask fitting."
            )

    occupancy_rows = (
        coarse_probe_df[
            coarse_probe_df[
                "target_kind"
            ]
            == "occupancy"
        ]
    )

    if len(
        occupancy_rows
    ):
        best_occupancy = (
            occupancy_rows
            .sort_values(
                "soft_dice_mean",
                ascending=False,
            )
            .iloc[0]
        )

        if int(
            best_occupancy[
                "token_cap"
            ]
        ) > 2048:
            diagnosis[
                "supported_architecture_changes"
            ].append(
                "With target construction controlled and short convergence used, a denser mask lattice improves the same dot-product head; "
                "attention token cap and mask resolution should be decoupled."
            )
        else:
            diagnosis[
                "not_yet_proven"
            ].append(
                "Higher coarse-mask lattice resolution did not outperform the corrected 2048 occupancy target; "
                "resolution alone is not currently a justified architecture change."
            )


# ---------------------------------------------------------------
# E / E2 — mask information
# ---------------------------------------------------------------

if len(loo_mask_df):
    loo_by_name = (
        loo_mask_df
        .set_index(
            "variant"
        )
    )

    required_loo = {
        "query_xyz",
        "explicit_all_query",
        "explicit_all_common_query",
        "explicit_all_shuffled",
    }

    if required_loo.issubset(
        set(
            loo_by_name.index
        )
    ):
        query_xyz = float(
            loo_by_name.loc[
                "query_xyz",
                "soft_dice_mean",
            ]
        )

        explicit = float(
            loo_by_name.loc[
                "explicit_all_query",
                "soft_dice_mean",
            ]
        )

        common_query = float(
            loo_by_name.loc[
                "explicit_all_common_query",
                "soft_dice_mean",
            ]
        )

        shuffled = float(
            loo_by_name.loc[
                "explicit_all_shuffled",
                "soft_dice_mean",
            ]
        )

        if (
            explicit
            > query_xyz
            + 0.05
            and explicit
            > shuffled
            + 0.05
        ):
            diagnosis[
                "supported_architecture_changes"
            ].append(
                "On leave-one-cell-out mask-head validation, correctly aligned explicit local spatial evidence beats query+XYZ and shuffled-evidence controls; "
                "an anchor-local spatial mask decoder is causally supported."
            )
        else:
            diagnosis[
                "not_yet_proven"
            ].append(
                "Leave-one-cell-out controls do not show a decisive advantage for correctly aligned explicit spatial evidence; "
                "high full-fit local-mask Dice may still reflect overfit/memorization."
            )

        if (
            common_query
            >= explicit
            - 0.05
        ):
            diagnosis[
                "confirmed_findings"
            ].append(
                "A common/identical query preserves most leave-one-cell-out local-mask performance; "
                "spatial anchor/evidence can carry instance identity without strongly distinct semantic query vectors."
            )

elif len(local_mask_df):
    diagnosis[
        "not_yet_proven"
    ].append(
        "Local mask full-fit completed but leave-one-cell-out validation did not; "
        "do not interpret high full-fit Dice as proof of spatial-information causality."
    )


if len(local_mask_df):
    local_by_name = (
        local_mask_df
        .set_index(
            "name"
        )
    )

    if {
        "explicit_all_query",
        "oracle_center_explicit_all_query",
    }.issubset(
        set(
            local_by_name.index
        )
    ):
        predicted_center_dice = float(
            local_by_name.loc[
                "explicit_all_query",
                "soft_dice_mean",
            ]
        )

        oracle_center_dice = float(
            local_by_name.loc[
                "oracle_center_explicit_all_query",
                "soft_dice_mean",
            ]
        )

        if (
            oracle_center_dice
            > predicted_center_dice
            + 0.05
        ):
            diagnosis[
                "supported_architecture_changes"
            ].append(
                "Oracle centers materially improve the same local mask decoder; "
                "center accuracy remains a direct mask bottleneck."
            )
        else:
            diagnosis[
                "confirmed_findings"
            ].append(
                "Once a rich local mask decoder sees a sufficiently large crop, oracle centers do not materially improve full-fit mask Dice; "
                "perfect center precision is not required for this local segmentation mechanism."
            )


# ---------------------------------------------------------------
# F — support radius
# ---------------------------------------------------------------

if len(support_summary):
    predicted_viable = (
        support_summary[
            support_summary[
                "pred_center_coverage_min"
            ]
            >= 0.99
        ]
        .sort_values(
            "radius_dref"
        )
    )

    if len(
        predicted_viable
    ):
        smallest = (
            predicted_viable.iloc[0]
        )

        smallest_radius = float(
            smallest[
                "radius_dref"
            ]
        )

        if smallest_radius < 2.5:
            diagnosis[
                "confirmed_findings"
            ].append(
                f"A {smallest_radius:.2f} dref radial support already covers at least 99% of every source-9 GT cell from the current matched predicted centers; "
                "2.5 dref rendering support is unnecessarily broad for this scene."
            )


diagnosis_path = (
    RUN_DIR
    / "diagnosis.json"
)

diagnosis_path.write_text(
    json.dumps(
        diagnosis,
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 92)
print(
    "NOTEBOOK 28B — CORRECTED CAUSAL DIAGNOSIS"
)
print("=" * 92)

print()
print("CONFIRMED FINDINGS")
for item in diagnosis[
    "confirmed_findings"
]:
    print("•", item)

print()
print("SUPPORTED ARCHITECTURE CHANGES")
for item in diagnosis[
    "supported_architecture_changes"
]:
    print("•", item)

print()
print("NOT YET PROVEN")
for item in diagnosis[
    "not_yet_proven"
]:
    print("•", item)

print()
print("Saved:", diagnosis_path)


## 7. Final comparison tables and run-health check

In [ ]:
print("Selection audit — MUST show 9 unique GT cells")
display(selection_audit)

print()
print("A — center pool")
display(center_audit)

print()
print("A — fixed nine paired centers")
display(paired_center_df)

if len(proposal_set_df):
    print()
    print("B — proposal reasoning")
    display(proposal_set_df)

if len(center_probe_df):
    print()
    print("C — center information controls")
    display(center_probe_df)

if len(coarse_probe_df):
    print()
    print("D — coarse target / resolution")
    display(coarse_probe_df)

if len(local_mask_df):
    print()
    print("E — local mask full-fit controls")
    display(local_mask_df)

if len(loo_mask_df):
    print()
    print("E2 — leave-one-cell-out mask controls")
    display(loo_mask_df)

if len(support_summary):
    print()
    print("F — support geometry")
    display(support_summary)

print()
print("Run directory:", RUN_DIR)

if ERRORS_JSONL.exists():
    print()
    print("WARNING: one or more phases failed.")
    print("Inspect:", ERRORS_JSONL)
else:
    print()
    print("All enabled phases completed without logged exceptions.")


## 8. Optional Napari center overlay

Set `OPEN_NAPARI_AT_END=True` before running if desired.

The viewer shows all 9 paired cells:

- GT centers,
- selected initial proposal-query anchors,
- current final query centers,
- best local-tensor+query center probe centers if Experiment C completed.


In [ ]:
if OPEN_NAPARI_AT_END:
    import napari

    source_voxels = np.argwhere(
        current_labels_native
        == SOURCE_ID
    )

    lo = source_voxels.min(
        axis=0
    )

    hi = (
        source_voxels.max(
            axis=0
        )
        + 1
    )

    margin_voxels = np.ceil(
        (
            2.0
            * dref_um
        )
        / spacing_native
    ).astype(int)

    lo = np.maximum(
        0,
        lo - margin_voxels,
    )

    hi = np.minimum(
        np.asarray(
            current_labels_native.shape
        ),
        hi + margin_voxels,
    )

    crop = tuple(
        slice(
            int(a),
            int(z),
        )
        for a, z
        in zip(
            lo,
            hi,
        )
    )

    raw_native = (
        batch_cpu[
            "spatial_inputs"
        ][0, 0]
        .detach()
        .cpu()
        .float()
        .numpy()
    )

    raw_crop = raw_native[
        crop
    ]

    full_shape = np.asarray(
        current_labels_native.shape,
        dtype=np.float64,
    )

    extent_um = (
        full_shape - 1
    ) * spacing_native

    def refs_to_crop_voxels(
        refs,
    ):
        refs = np.asarray(
            refs
        )

        global_voxels = (
            (
                refs
                * dref_um
                + 0.5
                * extent_um[None]
            )
            / spacing_native[None]
        )

        return (
            global_voxels
            - lo[None]
        )

    viewer = napari.Viewer(
        title=(
            "STIR-Net Notebook 28B "
            "— Corrected 9-cell Center Audit"
        )
    )

    scale = tuple(
        float(v)
        for v
        in spacing_native
    )

    viewer.add_image(
        raw_crop,
        name="Raw",
        scale=scale,
    )

    viewer.add_points(
        refs_to_crop_voxels(
            selected_gt_centers
            .numpy()
        ),
        name="GT centers",
        scale=scale,
        size=6,
    )

    viewer.add_points(
        refs_to_crop_voxels(
            selected_initial_refs
            .numpy()
        ),
        name="Selected initial centers",
        scale=scale,
        size=6,
    )

    viewer.add_points(
        refs_to_crop_voxels(
            selected_final_refs
            .numpy()
        ),
        name="Current final centers",
        scale=scale,
        size=6,
    )

    if (
        "local_tensor_plus_query"
        in center_probe_refs
    ):
        viewer.add_points(
            refs_to_crop_voxels(
                center_probe_refs[
                    "local_tensor_plus_query"
                ].numpy()
            ),
            name=(
                "Local-tensor+query "
                "refined centers"
            ),
            scale=scale,
            size=7,
        )

    viewer.dims.ndisplay = 3


### Output files

Notebook 28B writes only small diagnostics:

```text
00_source9_selection_audit.csv
A_center_pool_audit.csv
A_center_paired_9cells.csv
B_proposal_set_reasoning.csv
C_center_information_controls.csv
D_coarse_dot_mask_probe.csv
E_local_mask_information_full_fit.csv
E2_loo_mask_summary.csv
F_support_summary.csv
diagnosis.json
experiment.log
results.jsonl
errors.jsonl              # only if something failed
```

The `E_local_mask_predictions_small.npz` file contains only tiny 20³ probability cubes, not full native volumes.
